# 00 — Data smoke-test (Phase 0)

**Purpose: prove the whole pipeline runs end-to-end on one session before we touch the science.**

The HMM at the end is *not* a result. It is a canary. If it fits, then data access,
trial alignment, binning, and model fitting all work, and Phase 1 can start.

Steps:
1. Connect to public **openalyx** (no account needed).
2. Pull the BWM metadata tables; count good units per region.
3. Find sessions where our candidate-axis regions were recorded **simultaneously**.
4. *(gated)* Download one insertion's spikes + the session trials table.
5. Build a `(trials x time x neurons)` tensor aligned to stimulus onset.
6. Fit a Poisson HMM and relate the state sequence to reaction time.

Steps 1-3 are **metadata only** and cost nothing. Step 4 downloads ~440 MB.

Run with the `neurodatarehack` conda env.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# make `src` importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.access import get_one, sessions_table, units_table, region_cooccurrence, load_units, load_trials
from src.data.build_matrices import bin_spikes_by_event, stack_for_hmmlearn, drop_silent_units

PARAMS = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
AREAS = yaml.safe_load((ROOT / "config" / "areas.yaml").read_text())
AXIS = AREAS["candidate_axis"]

rng = np.random.default_rng(PARAMS["random_seed"])
print("candidate axis:", AXIS)

## 1. Connect to openalyx

`get_one()` uses ONE-api's built-in public credentials and pins the aggregate-table
tag so `bwm_units` reproduces the paper's frozen unit set. No registration needed.

In [ ]:
one = get_one(cache_dir=PARAMS["data"]["one_cache_dir"])
print("mode:      ", one.mode)
print("cache_dir: ", one.cache_dir)

## 2. Metadata tables and good-unit yields

`units_table` asserts a hash of the unit UUIDs against the `2023_12_bwm_release`
freeze, so these are exactly the units the paper analysed.

Expect **699 insertions / 459 sessions**, and **62,990 units over 210 Beryl regions**.

In [ ]:
bwm_df = sessions_table(one)
unit_df = units_table(one)

print(f"insertions: {bwm_df.pid.nunique()}   sessions: {bwm_df.eid.nunique()}")
print(f"good units: {len(unit_df)}   Beryl regions: {unit_df.Beryl.nunique()}")

yields = pd.DataFrame({
    "good_units": unit_df[unit_df.Beryl.isin(AXIS)].Beryl.value_counts().reindex(AXIS),
    "n_sessions": unit_df[unit_df.Beryl.isin(AXIS)].groupby("Beryl", observed=True).eid.nunique().reindex(AXIS),
})
yields

## 3. Which axis regions co-occur in one session?

Single-trial methods (HMM, rSLDS, geometry, flow) need **simultaneously recorded**
populations. Pooling across sessions into a pseudopopulation destroys the trial-by-trial
correlations these models exist to capture. So the unit of selection is the *session*.

In [ ]:
sel = AREAS["selection"]
co = region_cooccurrence(
    unit_df, AXIS,
    min_units_per_region=sel["min_units_per_region"],
    min_regions_per_session=sel["min_regions_per_session"],
)
print(f"{len(co)} sessions have >= {sel['min_regions_per_session']} axis regions "
      f"with >= {sel['min_units_per_region']} good units each\n")
for _, r in co.head(6).iterrows():
    print(f"  {r.eid}  n_reg={r.n_regions}  " + ", ".join(f"{a}:{b}" for a, b in zip(r.regions, r.units)))

In [ ]:
# Pick the richest session, and within it the insertion carrying the most units
# of a single target region. That region is our smoke-test population.
best = co.iloc[0]
EID = best.eid

cand = (unit_df[(unit_df.eid == EID) & (unit_df.Beryl.isin(AXIS))]
        .groupby(["pid", "Beryl"], observed=True).size()
        .reset_index(name="n").sort_values("n", ascending=False))
display(cand)

PID = cand.iloc[0]["pid"]
REGION = cand.iloc[0]["Beryl"]
N_EXPECTED = int(cand.iloc[0]["n"])
print(f"\nsmoke-test target -> eid={EID}\n                    pid={PID}\n"
      f"                    region={REGION} ({N_EXPECTED} good units)")

## 4. Download (gated)

> **This is the only cell that downloads neural data.**
> With `qc=1.0`, `load_good_units` fetches `passingSpikes.table.pqt` (a good-units-only
> spike table) plus small `clusters.*`/`channels.*` files — **not** the raw ephys.
> Roughly **150–280 MB per insertion**. Cached afterwards, so re-running is free.

Set `DOWNLOAD = True` to proceed.

In [ ]:
DOWNLOAD = False  # <-- flip to True to fetch data

if not DOWNLOAD:
    raise RuntimeError("Set DOWNLOAD = True to download ~150-280 MB for this insertion.")

spikes, clusters = load_units(one, PID, qc=PARAMS["data"]["min_qc"])
trials, mask = load_trials(
    one, EID,
    min_rt=PARAMS["trials"]["min_rt_s"],
    max_rt=PARAMS["trials"]["max_rt_s"],
)
print(f"spikes: {spikes['times'].size:,} from {clusters.shape[0]} good units")
print(f"trials: {mask.sum()} usable of {len(trials)} total")

### Select the units belonging to our Beryl region

`clusters['acronym']` holds the *fine-grained* Allen acronym (e.g. `SCig`, `SCiw`,
`VISp2/3`). Beryl is a coarser rollup (`SCm`, `VISp`). We must map before filtering,
otherwise `acronym == 'SCm'` silently matches nothing.

In [ ]:
from iblatlas.regions import BrainRegions

br = BrainRegions()
clusters = clusters.copy()
clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")

unit_idx = np.flatnonzero((clusters["Beryl"] == REGION).values)
print(f"{len(unit_idx)} units in {REGION} (metadata said {N_EXPECTED})")
print(clusters["Beryl"].value_counts().head(8).to_string())

## 5. Build the (trials x time x neurons) tensor

Aligned to `stimOn_times`. The trial mask is applied first — it enforces the paper's
own criteria (reaction time in range, no NaN in key events).

In [ ]:
B = PARAMS["binning"]
good_trials = trials[mask]
events = good_trials[B["align_event"]].values

# keep only spikes from our region's units, and re-index those units to 0..n-1
keep_spike = np.isin(spikes["clusters"], unit_idx)
remap = -np.ones(clusters.shape[0], dtype=np.int64)
remap[unit_idx] = np.arange(unit_idx.size)

counts, bin_centers = bin_spikes_by_event(
    spike_times=spikes["times"][keep_spike],
    spike_clusters=remap[spikes["clusters"][keep_spike]],
    n_clusters=unit_idx.size,
    event_times=events,
    pre_s=B["pre_s"], post_s=B["post_s"], bin_size_s=B["bin_size_s"],
)
counts, kept = drop_silent_units(counts, min_total_spikes=1)
print(f"tensor: {counts.shape}  (trials x bins x neurons), dtype={counts.dtype}")
print(f"window: {bin_centers[0]:+.3f} .. {bin_centers[-1]:+.3f} s around {B['align_event']}")
print(f"mean rate: {counts.sum() / (counts.shape[0]*counts.shape[1]*B['bin_size_s']*counts.shape[2]):.2f} sp/s")

In [ ]:
# Sanity: population PSTH should show a visual response after stimulus onset.
psth = counts.mean(axis=(0, 2)) / B["bin_size_s"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(bin_centers, psth, lw=1.5)
ax.axvline(0, color="k", ls="--", lw=1, label="stimulus onset")
ax.set(xlabel="time from stimOn (s)", ylabel="rate (sp/s)", title=f"{REGION} population PSTH")
ax.legend(frameon=False); fig.tight_layout()

## 6. Fit a Poisson HMM (the canary)

Poisson emissions are the right likelihood for **raw spike counts** — so we deliberately
feed unsmoothed integers. `lengths` tells `hmmlearn` that trials are independent
sequences, so it never learns a transition across a trial boundary.

`n_states` here is an arbitrary Phase-0 value. Choosing K by cross-validated
log-likelihood is a **Phase 2** job — do not read anything into this fit.

In [ ]:
from hmmlearn.hmm import PoissonHMM

K = PARAMS["hmm"]["n_states"]
X, lengths = stack_for_hmmlearn(counts)

model = PoissonHMM(n_components=K, n_iter=PARAMS["hmm"]["n_iter"], random_state=PARAMS["random_seed"])
model.fit(X, lengths)

print(f"converged: {model.monitor_.converged}   iters: {model.monitor_.iter}")
print(f"log-likelihood: {model.score(X, lengths):,.1f}")

states = model.predict(X, lengths).reshape(counts.shape[0], counts.shape[1])
print(f"state sequence: {states.shape} (trials x bins)")

In [ ]:
# Do the inferred states carry anything behavioural?
# Sort trials by reaction time; if states track the decision, structure should tilt.
rt = (good_trials["firstMovement_times"] - good_trials["stimOn_times"]).values
order = np.argsort(rt)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})

im = axes[0].imshow(states[order], aspect="auto", interpolation="nearest",
                    extent=[bin_centers[0], bin_centers[-1], len(order), 0], cmap="viridis")
axes[0].plot(rt[order], np.arange(len(order)) + 0.5, color="r", lw=1.2, label="first movement")
axes[0].axvline(0, color="w", ls="--", lw=1)
axes[0].set(xlabel="time from stimOn (s)", ylabel="trial (sorted by RT)",
            title=f"{REGION}: inferred HMM state, K={K}")
axes[0].legend(frameon=False, loc="lower right")
fig.colorbar(im, ax=axes[0], label="state")

occupancy = np.stack([(states == k).mean(axis=0) for k in range(K)])
for k in range(K):
    axes[1].plot(bin_centers, occupancy[k], label=f"state {k}")
axes[1].axvline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="time from stimOn (s)", ylabel="P(state)", title="state occupancy")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()

## What passing means

If every cell above ran, then the following are all verified on real data:
openalyx access → frozen unit set → simultaneous-region selection → spike download →
trial masking → stimulus-aligned binning → Poisson HMM fit → behavioural readout.

That is Phase 0 complete. **Nothing here is a scientific result.**

Next — **Phase 1 (area selection):** widen step 3 across all 43 qualifying sessions,
decide the final region set, and freeze it into `config/areas.yaml` under `selected`.